# 05 — System Prompts, Temperature & Structured Data

**7 Prompt Engineering Techniques (Anthropic's Official List):**

| # | Technique | What it does |
|---|-----------|-------------|
| 1 | Be clear and direct | Unambiguous instructions reduce guessing |
| 2 | Use examples (multishot) | Show 2-5 examples to guide format/reasoning |
| 3 | Let Claude think (CoT) | Step-by-step reasoning for complex tasks |
| 4 | Use XML tags | Structure prompt sections for reliable parsing |
| 5 | Give Claude a role (system prompts) | **→ This notebook, Section 1** |
| 6 | Prefill Claude's response | **→ This notebook, Section 3** |
| 7 | Chain complex prompts | Break multi-step tasks into sequential calls |

---

### This notebook covers 3 topics:

**1. System Prompts** — control *how* Claude responds (role, tone, constraints), not *what* it responds  
**2. Temperature** — control output randomness (deterministic ↔ creative)  
**3. Structured Data Generation** — force Claude to output clean, parseable JSON using prefill + stop sequences

---
## Setup

In [10]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()

---
## 1. System Prompts

**What is a system prompt?**  
A special message passed via the `system` parameter (not inside `messages`).  
It defines Claude's **role, behavior, and constraints** before any user interaction begins.

**Key distinction:**  
- System prompt controls **how** Claude responds (tone, format, boundaries)  
- User messages control **what** Claude responds to (the actual question)

**Example:** Same question "How do I solve 2x+3=7?"  
- Without system prompt → Claude gives the direct answer: "x = 2"  
- With math tutor system prompt → Claude guides step-by-step without revealing the answer

### 1.1 Helper Functions

Since the API is stateless (stores nothing between calls), we manually maintain a `messages` list and append to it.

In [22]:
def add_user_message(messages, content):
    new_message = {"role":"user", "content":content}
    messages.append(new_message) 

def add_assistant_message(messages, content):
    new_message = {"role":"assistant", "content":content}
    messages.append(new_message) 

### 1.2 Basic Implementation

The simplest approach: `system` is a required parameter in the `chat()` function.

**How it works:**
- `system` is passed as a plain string to `client.messages.create()`
- It sits outside the `messages` list — Claude sees it as a persistent instruction layer
- The system prompt is read **once** at the start and applies to the entire conversation

In [ ]:
# Basic: system_prompt is a required parameter
system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

def chat(model, max_tokens, messages, system_prompt):
    response = client.messages.create(
        model = model,
        max_tokens = max_tokens,
        messages = messages,
        system = system_prompt
    )
    return response.content[0].text

model = "claude-haiku-4-5-20251001"
max_tokens = 5000
messages = []
user_content = [{"type": "text", "text": "How do I solve 2x+3 = 7 for x?"}]

add_user_message(messages, user_content)
ans1 = chat(model, max_tokens, messages, system_prompt) 
print(ans1)

Great question! Let's work through this step by step.

Your equation is: 2x + 3 = 7

**Step 1:** What do you think you should do first to isolate the term with x in it?

*Hint: You have a +3 on the left side along with the 2x. How could you remove that +3?*

Once you figure out what to do in Step 1, let me know what you get, and we'll move to Step 2!


### 1.3 Flexible Implementation (Production Pattern)

**Problem with basic version:** `system_prompt` is required — every call *must* have one, even when you don't need it.

**Fix:** Make it optional with `system_prompt=None`, and use `**params` dict unpacking.

**Why this pattern matters:**  
- Production code needs to handle calls with and without system prompts
- The `params` dict pattern is extensible — easy to add `temperature`, `stop_sequences`, `tools`, etc. later without changing the function signature every time

In [19]:
# Flexible: system_prompt is optional, uses **params dict unpacking
def chat(model, max_tokens, messages, system_prompt=None):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
    }

    if system_prompt:
        params["system"] = system_prompt

    response = client.messages.create(**params)
    return response.content[0].text


add_user_message(messages, user_content)
ans2 = chat(model, max_tokens, messages, system_prompt) 
print("ans2: ", ans2)

ans2:  Great question! Let me guide you through this step-by-step.

**Step 1:** Look at the equation: 2x + 3 = 7

The goal is to get x by itself on one side. What do you think is in the way of x being alone on the left side?

**Step 2:** Once you identify what's getting in the way, think about the opposite operation you'd need to use. For example:
- If something is being added, you subtract it from both sides
- If something is being multiplied, you divide both sides by it

Can you identify what operation we need to do first, and what number we should use?

**Step 3:** Once you've done that operation to both sides, you should have 2x by itself on the left. Then we can deal with the 2 in front of x.

Why don't you give it a try and tell me what you get after the first step?


---
## 2. Temperature

**What does it do?**  
Controls the **randomness** of token selection during generation.

| Temperature | Behavior | Use case |
|-------------|----------|----------|
| `0.0` | Deterministic — always picks the highest-probability token | Factual answers, data extraction, JSON output |
| `0.5` | Moderate variety | General conversation |
| `1.0` (default) | Full randomness from the probability distribution | Creative writing, brainstorming |

**How it works under the hood:**  
After the model computes probabilities for the next token, temperature **scales** the distribution:  
- Low temperature → sharpens the distribution (top token dominates)  
- High temperature → flattens the distribution (more tokens get a chance)

**Rule of thumb:**  
- Need consistency/accuracy → low temperature (0.0–0.3)  
- Need creativity/variety → high temperature (0.7–1.0)

In [18]:
# Adding temperature to the flexible chat function
def chat_temp(model, max_tokens, messages, system_prompt=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature
    }

    if system_prompt:
        params["system"] = system_prompt

    response = client.messages.create(**params)
    return response.content[0].text


ans_temp01 = chat_temp(model, max_tokens, messages, system_prompt, temperature=1.0) 
ans_temp02 = chat_temp(model, max_tokens, messages, system_prompt, temperature=0.1) 
print("ans_temp01: ", ans_temp01)
print("----------------------------")
print("ans_temp02: ", ans_temp02)

ans_temp01:  Great question! Let me guide you through this step by step.

**Step 1:** Look at your equation: 2x + 3 = 7

Your goal is to get x by itself on one side. What do you think you should do first?

*Hint: Think about what's being added to the 2x. How could you remove the +3?*

Once you figure that out, tell me what operation you'd do to both sides of the equation!
----------------------------
ans_temp02:  Great question! Let me guide you through this step by step.

**Step 1:** Look at your equation: 2x + 3 = 7

Your goal is to get x by itself on one side. What do you think you should do first to isolate the term with x in it?

*Hint: Think about what's being added to 2x, and how you could remove it.*

Once you tell me what you'd do, we can move to the next step!


**Observation:** Both responses are similar in structure (same system prompt), but `temperature=1.0` has slightly more varied wording while `temperature=0.1` is more predictable and consistent. For this simple prompt the difference is subtle — the gap becomes much more obvious with creative tasks.

---
## 3. Structured Data Generation — Prefill + Stop Sequences

### 3.1 The Problem

When you ask Claude to generate JSON, it wraps the output in markdown code fences:

```
Here's the JSON you requested:
```json
{"name": "..."}
```
```

This breaks `json.loads()` because the response contains:
- Preamble text ("Here's the JSON...")
- Markdown delimiters (` ```json ` and ` ``` `)

### 3.2 The Solution: Prefill + Stop Sequence

**Two-part trick:**

1. **Prefill** — inject an assistant message containing ` ```json ` so Claude thinks it already started outputting JSON and continues from there
2. **Stop sequence** — set ` ``` ` as a stop sequence so Claude stops *before* writing the closing fence

**Result:** The response contains only the raw JSON — no preamble, no markdown. Directly parseable with `json.loads()`.

```
Without prefill+stop:  ```json\n{...}\n```    ← can't parse
With prefill+stop:     {...}                   ← clean JSON
```

### 3.3 Demo: Without Prefill + Stop (the problem)

In [31]:
# Updated chat function with stop_sequences support
def chat(model, max_tokens, messages, system_prompt=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
    }

    if system_prompt:
        params["system"] = system_prompt
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

# WITHOUT prefill + stop: response includes ```json ... ``` wrapper
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
ans_without_prefilling_stop = chat(model, max_tokens, messages, stop_sequences=None) 
print("ans_without_prefilling_stop: ", ans_without_prefilling_stop)

ans_without_prefilling_stop:  ```json
{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}
```


**↑ Problem:** Response starts with ` ```json ` and ends with ` ``` `. Calling `json.loads()` on this would fail.

### 3.4 Demo: With Prefill + Stop (the fix)

In [32]:
# WITH prefill + stop: clean JSON output
# Step 1: Prefill — add an assistant message with ```json to "start" the response
# Step 2: Stop — set ``` as stop sequence so it doesn't write the closing fence
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")
ans_with_prefilling_stop = chat(model, max_tokens, messages, stop_sequences=["```"]) 
print("ans_with_prefilling_stop: ", ans_with_prefilling_stop)

ans_with_prefilling_stop:  
{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1",
      "RoleArn": "arn:aws:iam::123456789012:role/EventBridgeRole"
    }
  ]
}



**↑ Clean output!** No ` ```json ` prefix, no ` ``` ` suffix. Just raw JSON.

### 3.5 Post-Processing: String → Dict

The API always returns a `str`. To use it as structured data, parse with `json.loads()`.  
Use `.strip()` to remove any leading/trailing whitespace.

In [36]:
import json

# Clean up and parse the JSON
clean_json = json.loads(ans_with_prefilling_stop.strip())
clean_json

{'Name': 'MySimpleRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1',
   'RoleArn': 'arn:aws:iam::123456789012:role/EventBridgeRole'}]}

In [37]:
# Before parsing: str → After parsing: dict
print(type(ans_with_prefilling_stop))  # str  (raw API response)
print(type(clean_json))                # dict (usable Python object)

<class 'str'>
<class 'dict'>


---
## Summary

| Topic | Key Takeaway | API Parameter |
|-------|-------------|---------------|
| **System Prompt** | Controls *how* Claude responds (role/tone/constraints). Passed as `system=` string, not inside `messages`. | `system` |
| **Temperature** | 0.0 = deterministic, 1.0 = creative. Use low for data extraction, high for brainstorming. | `temperature` |
| **Structured Output** | Prefill assistant message with ` ```json ` + set stop sequence ` ``` ` → clean JSON output. Always `json.loads()` the result. | `messages` (prefill) + `stop_sequences` |

**Evolution of the `chat()` function across this notebook:**

```
v1 (basic):    chat(model, max_tokens, messages, system_prompt)
v2 (flexible): chat(model, max_tokens, messages, system_prompt=None)        ← uses **params
v3 (+ temp):   chat_temp(..., temperature=1.0)                              ← adds temperature
v4 (+ stop):   chat(..., stop_sequences=None)                               ← adds stop_sequences
```

**Note:** The prefill+stop technique for structured output is the *simple* approach. A more reliable method using **Tool Use with `tool_choice`** is covered in a later notebook (forces JSON schema compliance).